In [2]:
from langchain.chains import LLMChain
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
import os
import re
from dotenv import load_dotenv
import json

In [5]:
from pydantic import BaseModel, Field
from typing import List, Optional

class ProductSuggestion(BaseModel):
    product_id: str = Field(
        description="Unique identifier for the product being suggested"
    )
    caption: str = Field(
        description="Brief explanation of why this product is being recommended"
    )

class ResponseContent(BaseModel):
    text: str = Field(
        description="Main response text containing product recommendations and information"
    )
    suggestions: List[ProductSuggestion] = Field(
        default=[],
        description="List of specific product suggestions related to the customer's query"
    )

class ResponseModel(BaseModel):
    response: ResponseContent = Field(
        description="Container for the response content and product suggestions"
    )

parser = JsonOutputParser(pydantic_object=ResponseModel)

In [18]:
# Load environment variables
load_dotenv()
GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY")
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=GOOGLE_API_KEY)
# Prompt for gathering user information
interaction_prompt = PromptTemplate(
    template="""
            your work in cosmetic company and you are a sales agent and you are selling products to customers .
            
            the doctor will give you same prodect and his suggessions and list of products and their details base on the user needs and your work to restructure the doctor suggessions in the clear json output structure
        
        the output should be in the following json structure:
        {format_instructions}
        
        doctor suggessions:
        {doctor_suggesions}
        
        IMPORTANT:
            please answer in the same language of doctor suggessions
    """,
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

interaction_chain = interaction_prompt | llm | parser





In [30]:
doctor_suggesions

[ToolMessage(content=[], name='query_products', id='af1f7bee-b2a8-44ca-8ff2-e7e93dda8b10', tool_call_id='956e354f-b294-476d-a08d-50fc07c96649'),
 AIMessage(content='أكيد يا فندم، أنا هنا عشان أساعدك. عشان نقدر نلاقي أفضل حل لتساقط الشعر الدهني، محتاجين نعرف شوية تفاصيل زيادة. ممكن توضحيلي:\n\n1.  **نوع التساقط:** هل التساقط بكميات كبيرة ولا تدريجي؟ وهل فيه أي مناطق معينة في فروة الرأس فيها فراغات؟\n2.  **المنتجات المستخدمة حالياً:** إيه أنواع الشامبو والبلسم اللي بتستخدميها؟ وهل بتستخدمي أي منتجات تانية زي زيوت أو كريمات؟\n3.  **الحالة الصحية:** هل فيه أي مشاكل صحية أو أأدوية بتاخديها ممكن تكون سبب في التساقط؟\n\nوبناءً على المعلومات دي، هقدر أرشحلك مجموعة منتجات Nileva المتخصصة في علاج تساقط الشعر الدهني، واللي بتحتوي على مكونات طبيعية زي الروزماري وزيت شجرة  الشاي، اللي بتساعد على تقوية بصيلات الشعر وتحسين صحة فروة الرأس.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-6b1

In [31]:

response = interaction_chain.invoke({
                                     "doctor_suggesions" : doctor_suggesions }
                                     )



In [32]:
response

{'response': {'text': 'أكيد يا فندم، أنا هنا عشان أساعدك. عشان نقدر نلاقي أفضل حل لتساقط الشعر الدهني، محتاجين نعرف شوية تفاصيل زيادة. ممكن توضحيلي:\n\n1.  **نوع التساقط:** هل التساقط بكميات كبيرة ولا تدريجي؟ وهل فيه أي مناطق معينة في فروة الرأس فيها فراغات؟\n2.  **المنتجات المستخدمة حالياً:** إيه أنواع الشامبو والبلسم اللي بتستخدميها؟ وهل بتستخدمي أي منتجات تانية زي زيوت أو كريمات؟\n3.  **الحالة الصحية:** هل فيه أي مشاكل صحية أو أأدوية بتاخديها ممكن تكون سبب في التساقط؟\n\nوبناءً على المعلومات دي، هقدر أرشحلك مجموعة منتجات Nileva المتخصصة في علاج تساقط الشعر الدهني، واللي بتحتوي على مكونات طبيعية زي الروزماري وزيت شجرة  الشاي، اللي بتساعد على تقوية بصيلات الشعر وتحسين صحة فروة الرأس.',
  'suggestions': []}}